# 1. A polyethylene melt from SMILES to minimized coordinates

The all-atom PhantomWalk workflow in four FlowerMD calls, followed by the
hand-off this repository adds: a periodic Sage 2.3.0 minimization at the
target density and the paper's quality metric, the energy the minimizer
removed per atom in units of Sage's largest Lennard-Jones well depth.

Runs in seconds on a GPU.

In [1]:
import hoomd

# GPU if one is visible, otherwise CPU. OpenMM follows the same choice.
try:
    DEVICE = hoomd.device.GPU()
    OPENMM_PLATFORM = "CUDA"
except Exception:
    DEVICE = hoomd.device.CPU()
    OPENMM_PLATFORM = "CPU"
print(DEVICE, OPENMM_PLATFORM)

<hoomd.device.GPU object at 0x14625a9d5730> CUDA


In [2]:
# The frozen all-atom PhantomWalk protocol, written out explicitly.
PROTOCOL = dict(
    bonded="uff", A=1250.0, gamma=200.0, kT=1.0, r_cut=3.5, bonded_scale=30.0,
    epsilon_weighting=True, protect_stereochemistry=True, stereo_k=30000.0,
)
RUN = dict(
    dpd_min_steps=3500, dpd_chunk=500, dpd_max_steps=40000, energy_tol=0.02,
    consecutive=2, dpd_samples_per_chunk=5, fire_steps=100, fire_dt=0.001,
    require_convergence=False,
)
DT = 0.001

In [3]:
import unyt as u
from flowermd.library import AllAtomDPD, AllAtomLattice, AllAtomPhantomWalk, PolyEthylene
from phantomwalk.all_atom import sage_handoff

chains = PolyEthylene(lengths=50, num_mols=12)          # 12 chains of 100 carbons
system = AllAtomLattice(chains, density=0.85 * u.g / u.cm**3, unit="repeat", seed=1)
ff = AllAtomDPD(system.system, **PROTOCOL)
sim = AllAtomPhantomWalk.from_system(system, forcefield=ff, dt=DT, device=DEVICE, seed=1)
record = sim.run_initialization(**RUN)
print(f"{record['n_particles']} atoms, DPD stationary: {record['dpd_converged']} "
      f"after {record['dpd_steps']} steps, {record['timings_s']['total']:.1f} s in total")

Initializing simulation state from a gsd.hoomd.Frame.
Step 5500 of 40000; TPS: 0.0; ETA: nan hours, nan minutes
3624 atoms, DPD stationary: True after 9000 steps, 3.8 s in total


In [4]:
import numpy as np

box_nm = np.asarray(ff.frame.configuration.box[:3]) / 10
handoff, minimized_a = sage_handoff(sim.to_compound(), box_nm, platform=OPENMM_PLATFORM)
print(f"Sage energy removed: {handoff['energy_removed_sage_epsilon_atom']:.2f} eps_max per atom "
      f"(finite: {handoff['finite']}, minimizer {handoff['minimizer_s']:.1f} s)")

Sage energy removed: 3.74 eps_max per atom (finite: True, minimizer 0.7 s)


`record` holds every setting, the wall time of each phase, the DPD energy
history and the software versions; `sim.write_record("record.json")` saves it.